# 01 — Translation Pipeline Overview

Descriptive statistics on the full pipeline output across all languages and services.
This notebook answers the basic questions before analysis:
- How many languages have translations from each service?
- What is the coverage gap across the 185 ISO 639-1 languages?
- Which services consistently fail on which language families?
- Where does the pipeline produce translations vs. fall silent?

**Run order:** This notebook should be run first. It does not depend on the evaluation scripts.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from data_generation_scripts.utils import get_data_directory_path, read_csv_file

DATA_DIR = get_data_directory_path()
TERMS = ['Computational Humanities', 'Digital Humanities']
VARIANTS = ['comparative', 'minimal', 'expert_persona', 'contextual', 'native_rationale']

SERVICE_COLS = {
    'Google Translate': 'gt_translated_term',
    'EasyNMT':          'enmt_translated_term',
    'Wikipedia':        'wikipedia_translated_term',
    'OpenAI':           'openai_translated_term',
    'Claude':           'claude_translated_term',
    'Ollama':           'ollama_translated_term',
}

print(f'Data directory: {DATA_DIR}')

## 1.1 Load Pipeline Outputs

In [ ]:
def load_all_variants(data_dir, term, variants=VARIANTS):
    """Load and concatenate all variant CSVs for a term."""
    dfs = []
    term_slug = term.lower().replace(' ', '_')
    for variant in variants:
        if variant == 'comparative':
            path = os.path.join(data_dir, 'metadata_files', 'translated_terms',
                                term_slug, 'initial_translated_terms.csv')
        else:
            path = os.path.join(data_dir, 'metadata_files', 'translated_terms',
                                term_slug, 'prompt_variants',
                                f'{variant}_initial_translated_terms.csv')
        if os.path.exists(path):
            df = read_csv_file(path)
            df['prompt_variant'] = variant
            df['term_source_query'] = term
            dfs.append(df)
        else:
            print(f'  ⚠ Missing: {path}')
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

all_dfs = {term: load_all_variants(DATA_DIR, term) for term in TERMS}
for term, df in all_dfs.items():
    print(f'{term}: {len(df)} rows across {df["prompt_variant"].nunique()} variants')

## 1.2 Coverage by Service

How many of the 185 languages have a translation from each service?

In [ ]:
for term, df in all_dfs.items():
    # Use comparative variant as baseline (has all primary services)
    baseline = df[df['prompt_variant'] == 'comparative']
    total = baseline['language_code'].nunique()
    print(f'\n{term} — {total} languages in baseline')
    for service, col in SERVICE_COLS.items():
        if col in baseline.columns:
            n = baseline[col].notna().sum()
            pct = round(n / total * 100, 1)
            bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
            print(f'  {service:20s} {bar} {n:3d}/{total} ({pct}%)')
        else:
            print(f'  {service:20s} (column not present)')

## 1.3 Coverage Heatmap by Language Family

In [ ]:
LANGUAGE_FAMILIES = {
    'Romance':     ['es', 'fr', 'it', 'pt', 'ro', 'ca', 'gl'],
    'Germanic':    ['de', 'nl', 'sv', 'da', 'no', 'af'],
    'Slavic':      ['ru', 'pl', 'cs', 'sk', 'bg', 'hr', 'sr', 'uk', 'sl'],
    'East Asian':  ['zh', 'ja', 'ko'],
    'Semitic':     ['ar', 'he'],
    'South Asian': ['hi', 'bn', 'ur', 'ta', 'te'],
    'Other':       [],
}

def get_family(code):
    for fam, codes in LANGUAGE_FAMILIES.items():
        if code in codes: return fam
    return 'Other'

term = TERMS[0]
baseline = all_dfs[term][all_dfs[term]['prompt_variant'] == 'comparative'].copy()
baseline['language_family'] = baseline['language_code'].apply(get_family)

# Build coverage matrix
rows = []
for family in LANGUAGE_FAMILIES:
    fam_df = baseline[baseline['language_family'] == family]
    if len(fam_df) == 0: continue
    row = {'Family': family, 'N': len(fam_df)}
    for service, col in SERVICE_COLS.items():
        if col in fam_df.columns:
            row[service] = round(fam_df[col].notna().mean() * 100, 1)
        else:
            row[service] = 0.0
    rows.append(row)

coverage_df = pd.DataFrame(rows).set_index('Family')
n_col = coverage_df.pop('N')

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(coverage_df, annot=True, fmt='.0f', cmap='YlGn',
            vmin=0, vmax=100, ax=ax, cbar_kws={'label': '% coverage'})
ax.set_title(f'Service Coverage by Language Family — {term}', pad=12)
ax.set_xlabel('Translation Service')
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'metadata_files', 'evaluation', '01_coverage_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.show()
coverage_df

## 1.4 Languages with Zero Coverage

These are the most interesting cases for the paper — languages where the pipeline
produced nothing at all. Are they low-resource languages? Script-diverse languages?
Languages where 'Digital Humanities' genuinely has no circulation?

In [ ]:
for term, df in all_dfs.items():
    baseline = df[df['prompt_variant'] == 'comparative'].copy()
    service_columns = [col for col in SERVICE_COLS.values() if col in baseline.columns]
    baseline['n_services'] = baseline[service_columns].notna().sum(axis=1)
    zero_coverage = baseline[baseline['n_services'] == 0]
    low_coverage = baseline[baseline['n_services'].between(1, 2)]
    
    print(f'\n{term}:')
    print(f'  Zero coverage ({len(zero_coverage)} languages):')
    lang_col = 'language_name' if 'language_name' in zero_coverage.columns else 'language_code'
    for _, row in zero_coverage.iterrows():
        print(f'    {row["language_code"]:5s} {row.get(lang_col, "")}')
    print(f'  Low coverage (1-2 services, {len(low_coverage)} languages):')
    for _, row in low_coverage.head(10).iterrows():
        print(f'    {row["language_code"]:5s} {row.get(lang_col, "")} — {int(row["n_services"])} services')

## 1.5 Error Log Analysis

What do the error logs tell us about *why* translations failed?

In [ ]:
error_dir = os.path.join(DATA_DIR, 'error_logs', 'translation_files')
if os.path.exists(error_dir):
    error_files = [f for f in os.listdir(error_dir) if f.endswith('.csv')]
    for fname in error_files:
        path = os.path.join(error_dir, fname)
        try:
            edf = read_csv_file(path)
            service = fname.replace('_translation_errors.csv', '')
            print(f'\n{service}: {len(edf)} errors')
            if 'status_code' in edf.columns:
                print(edf['status_code'].value_counts().to_string())
            if 'error_url' in edf.columns:
                print(edf['error_url'].value_counts().head(5).to_string())
        except Exception as e:
            print(f'  Could not read {fname}: {e}')
else:
    print(f'Error log directory not found: {error_dir}')

## 1.6 Translation Length Distribution

Do translations cluster in length? Very short translations may be transliterations;
very long ones may be definitions rather than term equivalents.

In [ ]:
term = TERMS[0]
baseline = all_dfs[term][all_dfs[term]['prompt_variant'] == 'comparative'].copy()

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
axes = axes.flatten()

for i, (service, col) in enumerate(SERVICE_COLS.items()):
    if col not in baseline.columns or i >= len(axes): continue
    vals = baseline[col].dropna().astype(str)
    lengths = vals.str.split().str.len()
    axes[i].hist(lengths, bins=range(1, 12), color='steelblue', edgecolor='white')
    axes[i].set_title(service)
    axes[i].set_xlabel('Word count')
    axes[i].set_ylabel('N languages')
    axes[i].axvline(lengths.median(), color='red', linestyle='--', label=f'median={lengths.median():.1f}')
    axes[i].legend(fontsize=8)

plt.suptitle(f'Translation Word Count Distribution — {term}', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'metadata_files', 'evaluation', '01_length_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()